# Lista 02 — Manipulação de dados

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

Referente ao módulo **02 · Manipulação de Dados**

---


Esta lista usa os dados reais da pasta `data/`: cinco anos de cotações de oito ações da
B3, indicadores macroeconômicos do Banco Central e uma base fictícia de clientes de
corretora.

Ela cobre NumPy, seleção e filtros em pandas, agrupamentos, junções e limpeza de dados.


**14 exercícios** (3 de teoria, 11 de código) ·
**Tempo estimado:** 2h30

> **Atenção — Use este arquivo depois de tentar.** As soluções aqui são uma entre várias
> possíveis — se a sua for diferente e funcionar, ela também está certa. O que importa
> é você conseguir explicar cada linha do que escreveu.

Cada exercício traz a solução comentada e, quando cabe, uma observação sobre o porquê
daquela escolha.

## Preparação

### Antes de começar — se você está no Google Colab

Este notebook lê arquivos da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "05_Exercicios"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

acoes = pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"])
empresas = pd.read_csv("../data/empresas_b3.csv")
clientes = pd.read_csv("../data/clientes_corretora.csv")
indicadores = pd.read_csv("../data/indicadores_macro.csv", parse_dates=["data"])

print("acoes      :", acoes.shape)
print("empresas   :", empresas.shape)
print("clientes   :", clientes.shape)
print("indicadores:", indicadores.shape)

---

## Exercício 1 (código) — Retornos com NumPy


Dado o array de preços abaixo, **sem usar loops**:

1. calcule o array de retornos diários simples, em porcentagem;
2. imprima o retorno médio, o desvio padrão amostral e o retorno acumulado do período;
3. confira que o retorno acumulado bate com a variação entre o primeiro e o último preço.


> **Dica:** `precos[1:]` é a série a partir do segundo dia; `precos[:-1]` vai até o penúltimo. Para o acumulado, `np.cumprod(1 + retornos)`.

In [ ]:
precos = np.array([32.50, 32.80, 31.90, 33.10, 33.45,
                   32.95, 34.20, 34.05, 33.60, 35.10])

retornos = (precos[1:] - precos[:-1]) / precos[:-1] * 100

print("Retornos diários (%):", np.round(retornos, 2))
print()
print(f"Retorno médio       : {retornos.mean():.3f}%")
print(f"Desvio padrão       : {retornos.std(ddof=1):.3f}%")

acumulado = (np.cumprod(1 + retornos / 100)[-1] - 1) * 100
print(f"Retorno acumulado   : {acumulado:.2f}%")
print(f"Conferindo pelo preço: {(precos[-1] / precos[0] - 1) * 100:.2f}%")

`ddof=1` calcula o desvio padrão **amostral** (divide por n−1). É o padrão em finanças,
porque os retornos observados são uma amostra do processo, não a população inteira. O
NumPy usa `ddof=0` por padrão; o pandas usa `ddof=1`. Essa diferença entre as duas
bibliotecas já causou muita confusão — vale saber que existe.

---

## Exercício 2 (código) — Máscaras booleanas


Usando o mesmo array de retornos do exercício anterior, responda **com código**:

1. quantos dias foram de alta?
2. qual a proporção de dias de alta?
3. qual o retorno médio dos dias de alta e o dos dias de baixa?
4. quais foram o melhor e o pior dia (posição e valor)?


> **Dica:** Um array booleano somado conta os `True`. `argmax` e `argmin` devolvem posições.

In [ ]:
precos = np.array([32.50, 32.80, 31.90, 33.10, 33.45,
                   32.95, 34.20, 34.05, 33.60, 35.10])
retornos = (precos[1:] - precos[:-1]) / precos[:-1] * 100

print("Dias de alta        :", (retornos > 0).sum())
print(f"Proporção de altas  : {(retornos > 0).mean():.1%}")
print(f"Média nos dias de alta : {retornos[retornos > 0].mean():.3f}%")
print(f"Média nos dias de baixa: {retornos[retornos < 0].mean():.3f}%")
print(f"Melhor dia: posição {retornos.argmax()} ({retornos.max():.2f}%)")
print(f"Pior dia  : posição {retornos.argmin()} ({retornos.min():.2f}%)")

`(retornos > 0).mean()` funciona porque `True` vale 1 e `False` vale 0 — a média de zeros
e uns é exatamente a proporção de `True`. É um truque que aparece o tempo todo.

---

## Exercício 3 (teoria) — O parâmetro `axis`


Um array `matriz` tem formato `(4, 3)`: quatro dias nas linhas, três ativos nas colunas.

1. O que `matriz.mean(axis=0)` calcula? Qual o formato do resultado?
2. E `matriz.mean(axis=1)`?
3. Qual dos dois você usaria para descobrir o preço médio de cada ativo no período?

**Resposta:**


1. `matriz.mean(axis=0)` percorre as **linhas** (os dias) e produz **um valor por
   coluna** — ou seja, a média de cada ativo ao longo dos quatro dias. O resultado tem
   formato `(3,)`.

2. `matriz.mean(axis=1)` percorre as **colunas** (os ativos) e produz **um valor por
   linha** — a média dos três ativos em cada dia. Formato `(4,)`.

3. **`axis=0`**, porque queremos um número por ativo (por coluna).

O truque para não errar: `axis` indica o eixo que **desaparece** no resultado. Com
`axis=0`, a dimensão das linhas some e sobram as colunas.

---

## Exercício 4 (código) — Primeiro contato com uma base


Usando o DataFrame `acoes` carregado na preparação, responda **com código**:

1. quantas linhas e colunas ele tem?
2. qual o período coberto (primeira e última data)?
3. quais ativos existem, e quantos pregões cada um tem?
4. há valores faltantes? em quais colunas?
5. qual o maior e o menor `fechamento_ajustado` da base inteira?

In [ ]:
print("1. Formato:", acoes.shape)

print("2. Período:", acoes["data"].min().date(), "->", acoes["data"].max().date())

print("\n3. Pregões por ativo:")
print(acoes["ticker"].value_counts())

print("\n4. Faltantes por coluna:")
print(acoes.isna().sum())

print("\n5. Fechamento ajustado:")
print(f"   máximo: R$ {acoes['fechamento_ajustado'].max():,.2f}")
print(f"   mínimo: R$ {acoes['fechamento_ajustado'].min():,.2f}")

Esses cinco comandos são o ritual de abertura de qualquer base. Fazê-los **antes** de
qualquer análise evita a situação clássica de descobrir, depois de três horas de trabalho,
que metade das linhas estava vazia.

---

## Exercício 5 (teoria) — `.loc` e `.iloc`


Um DataFrame `df` tem índice `["a", "b", "c", "d"]` e colunas `["x", "y", "z"]`.

1. O que devolve `df.loc["b", "y"]`?
2. O que devolve `df.iloc[1, 1]`?
3. `df.iloc[0:2]` devolve quantas linhas? E `df.loc["a":"c"]`?
4. Em qual situação você **precisa** usar `.iloc`?

**Resposta:**


1. O valor na linha de rótulo `"b"`, coluna `"y"`.
2. O valor na **posição** linha 1, coluna 1 — que, neste caso, é exatamente o mesmo valor
   do item anterior, porque a linha `"b"` está na posição 1 e a coluna `"y"` também.
3. `df.iloc[0:2]` devolve **duas** linhas (posições 0 e 1 — o fim é excluído).
   `df.loc["a":"c"]` devolve **três** linhas (a, b e c — com rótulos, as duas pontas são
   incluídas).
4. Quando o critério é **posicional** e não depende do conteúdo: pegar a primeira linha
   de um grupo, a última observação de uma série (`.iloc[-1]`), ou percorrer linhas por
   posição. Para tudo que é "esta coluna" ou "as linhas que satisfazem tal condição",
   use `.loc`.

---

## Exercício 6 (código) — Selecionando dados


Com o DataFrame `acoes`:

1. selecione apenas as colunas `data`, `ticker` e `fechamento_ajustado`, e mostre as 5
   primeiras linhas;
2. mostre a última linha da base (use posição);
3. mostre as linhas da WEGE3 com `data` e `fechamento_ajustado`, usando `.loc` com uma
   máscara booleana;
4. crie um DataFrame indexado por `data` contendo só a PETR4, e selecione todo o mês de
   março de 2025.

In [ ]:
# 1
print(acoes[["data", "ticker", "fechamento_ajustado"]].head())

# 2
print("\nÚltima linha:")
print(acoes.iloc[-1])

# 3
print("\nWEGE3:")
print(acoes.loc[acoes["ticker"] == "WEGE3", ["data", "fechamento_ajustado"]].head())

# 4
petr = acoes[acoes["ticker"] == "PETR4"].set_index("data")
print("\nPETR4 em março de 2025:")
print(petr.loc["2025-03", ["fechamento_ajustado"]].head())
print("Pregões no mês:", len(petr.loc["2025-03"]))

O item 4 mostra o poder de um índice de datas: `.loc["2025-03"]` seleciona o mês inteiro
sem nenhum filtro explícito. Isso só funciona porque a coluna foi lida com
`parse_dates=["data"]` e depois virou índice.

---

## Exercício 7 (código) — Filtros compostos


Responda com filtros sobre `acoes`:

1. quantos pregões da VALE3 tiveram fechamento ajustado acima de R$ 70?
2. quantas linhas são de bancos (ITUB4, BBDC4) **em 2023**?
3. quantas linhas **não** são de PETR4 nem de VALE3?
4. mostre os 5 pregões de maior volume da base, com data, ticker e volume.


> **Dica:** Para o item 2, extraia o ano com `acoes["data"].dt.year`. Para o 3, use `~` com `isin`.

In [ ]:
# 1
vale_alta = acoes[(acoes["ticker"] == "VALE3") & (acoes["fechamento_ajustado"] > 70)]
print("1.", len(vale_alta), "pregões")

# 2
bancos_2023 = acoes[
    acoes["ticker"].isin(["ITUB4", "BBDC4"]) & (acoes["data"].dt.year == 2023)
]
print("2.", len(bancos_2023), "linhas")

# 3
outros = acoes[~acoes["ticker"].isin(["PETR4", "VALE3"])]
print("3.", len(outros), "linhas")

# 4
print("\n4. Maiores volumes:")
print(acoes.nlargest(5, "volume")[["data", "ticker", "volume"]])

Cada condição precisa de parênteses próprios: `(a) & (b)`. Sem eles, o Python aplica `&`
antes das comparações e devolve um erro difícil de entender. E use `&` / `|` / `~`, nunca
`and` / `or` / `not` — estes esperam um único valor booleano, não uma coluna inteira.

---

## Exercício 8 (código) — Criando colunas


Acrescente a `acoes` as seguintes colunas:

1. `ano` e `mes`, extraídos da data;
2. `amplitude_dia`: `maxima - minima`;
3. `amplitude_pct`: a amplitude como porcentagem do fechamento;
4. `retorno_diario`: a variação percentual do `fechamento_ajustado` **dentro de cada
   ativo** (atenção: o cálculo não pode atravessar de um ticker para outro);
5. `tipo_dia`: `"alta"` se o retorno foi positivo, `"baixa"` caso contrário.

Mostre as 5 primeiras linhas com as colunas novas.


> **Dica:** Para o item 4: ordene por ticker e data, e use `acoes.groupby("ticker")["coluna"].pct_change()`.

In [ ]:
acoes = acoes.sort_values(["ticker", "data"])

acoes["ano"] = acoes["data"].dt.year
acoes["mes"] = acoes["data"].dt.month
acoes["amplitude_dia"] = acoes["maxima"] - acoes["minima"]
acoes["amplitude_pct"] = acoes["amplitude_dia"] / acoes["fechamento"] * 100
acoes["retorno_diario"] = acoes.groupby("ticker")["fechamento_ajustado"].pct_change() * 100
acoes["tipo_dia"] = np.where(acoes["retorno_diario"] > 0, "alta", "baixa")

acoes[["data", "ticker", "amplitude_pct", "retorno_diario", "tipo_dia"]].head()

O `groupby("ticker")` no item 4 é essencial. Sem ele, o primeiro pregão da VALE3 seria
comparado com o último da PETR4, gerando um "retorno" absurdo na fronteira entre os
ativos. Esse erro é silencioso: nada quebra, só o resultado fica errado. Sempre que
calcular variação em uma base empilhada, pergunte-se **dentro de qual grupo** o cálculo
deve acontecer.

---

## Exercício 9 (código) — Agrupando


Usando `groupby` com **agregação nomeada**, monte uma tabela com uma linha por ativo e as
colunas:

- `pregoes` — número de pregões;
- `retorno_medio` — retorno diário médio;
- `volatilidade` — desvio padrão do retorno diário;
- `melhor_dia` e `pior_dia`;
- `volume_medio`.

Ordene por volatilidade, da menor para a maior.

In [ ]:
resumo = acoes.groupby("ticker").agg(
    pregoes=("data", "count"),
    retorno_medio=("retorno_diario", "mean"),
    volatilidade=("retorno_diario", "std"),
    melhor_dia=("retorno_diario", "max"),
    pior_dia=("retorno_diario", "min"),
    volume_medio=("volume", "mean"),
).round(3)

resumo.sort_values("volatilidade")

A agregação nomeada (`nome=("coluna", "função")`) é preferível a `.agg(["mean", "std"])`
porque os nomes das colunas do resultado ficam legíveis e você não precisa lidar com
colunas de dois níveis depois.

---

## Exercício 10 (código) — Tabela dinâmica


Monte uma `pivot_table` com **ativos nas linhas**, **anos nas colunas** e o **retorno
diário médio** nas células, arredondado para 3 casas.

Depois responda em um comentário: em qual ano a maior parte dos ativos teve retorno médio
negativo?

In [ ]:
tabela = acoes.pivot_table(
    index="ticker", columns="ano", values="retorno_diario", aggfunc="mean"
).round(3)

print(tabela)

print("\nAtivos com retorno médio negativo em cada ano:")
print((tabela < 0).sum())

A última linha mostra um recurso muito útil: `(tabela < 0)` cria uma tabela de
`True`/`False`, e `.sum()` conta os `True` por coluna. É a mesma ideia das máscaras
booleanas do NumPy, aplicada a um DataFrame inteiro.

---

## Exercício 11 (código) — Cruzando tabelas


1. junte `acoes` com `empresas` pela coluna `ticker`, mantendo todas as linhas de `acoes`
   e validando que a relação é de muitos-para-um;
2. confirme que o número de linhas **não** mudou;
3. calcule o retorno diário médio **por setor**;
4. conte quantos ativos existem em cada setor.


> **Dica:** `how="left"` e `validate="m:1"`.

In [ ]:
linhas_antes = len(acoes)

completo = acoes.merge(empresas, on="ticker", how="left", validate="m:1")

print("Linhas antes :", linhas_antes)
print("Linhas depois:", len(completo))
print("Colunas novas:", [c for c in completo.columns if c not in acoes.columns])

print("\nRetorno diário médio por setor:")
print(completo.groupby("setor")["retorno_diario"].mean().round(4).sort_values())

print("\nAtivos por setor:")
print(completo.groupby("setor")["ticker"].nunique().sort_values(ascending=False))

Conferir o número de linhas antes e depois é obrigatório em qualquer merge. Se ele
aumentou, a chave está duplicada do lado direito e todas as somas seguintes estarão
infladas — um erro que não gera aviso nenhum. O `validate="m:1"` transforma esse risco
silencioso em um erro explícito.

---

## Exercício 12 (teoria) — Escolhendo o tipo de junção


Para cada situação, diga qual `how` você usaria em um `merge` e por quê:

1. Você tem a base de todos os clientes e quer acrescentar o saldo de quem investe. Nem
   todo cliente investe, e você quer manter todos na análise.
2. Você quer analisar apenas os pregões em que existe dado macroeconômico disponível.
3. Você quer descobrir quais tickers estão na tabela de preços mas **não** estão no
   cadastro de empresas.

**Resposta:**


1. **`how="left"`**, com a base de clientes à esquerda. Todos os clientes permanecem;
   quem não investe fica com `NaN` nas colunas de saldo — o que é a informação correta
   ("não tem saldo"), e não motivo para excluir a pessoa da análise.

2. **`how="inner"`**. Só interessam as linhas presentes nas duas tabelas.

3. **`how="left"` com `indicator=True`** (ou `how="outer"` com `indicator=True`), filtrando
   depois as linhas cujo `_merge` é `"left_only"`. Esses são exatamente os tickers sem
   correspondência no cadastro. É a forma padrão de auditar uma junção antes de confiar
   nela.

---

## Exercício 13 (código) — Limpando a base de clientes


A base `clientes` está suja. Faça, sobre uma **cópia** dela:

1. remova as linhas totalmente duplicadas e informe quantas foram;
2. converta `patrimonio_investido` para número (o formato é brasileiro, com `R$`, ponto
   de milhar e vírgula decimal) e informe quantos valores não converteram;
3. converta `data_cadastro` para data — atenção: há **dois** formatos misturados;
4. padronize `perfil_investidor` (espaços e caixa);
5. converta `ativo` para booleano;
6. transforme idades fora de [18, 110] em `NaN` e informe quantas eram.

Ao final, mostre `info()` e confirme que os tipos estão corretos.


> **Dica:** Para as datas, tente cada formato com `errors="coerce"` e combine os resultados com `.fillna()`.

In [ ]:
limpo = clientes.copy()

# 1. duplicatas
antes = len(limpo)
limpo = limpo.drop_duplicates()
print(f"1. {antes - len(limpo)} linhas duplicadas removidas")

# 2. dinheiro em texto -> número
texto = (
    limpo["patrimonio_investido"].astype(str).str.strip()
    .str.replace("R$", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)
limpo["patrimonio_investido"] = pd.to_numeric(texto, errors="coerce")
print(f"2. {limpo['patrimonio_investido'].isna().sum()} valores sem patrimônio")

# 3. datas em dois formatos
iso = pd.to_datetime(limpo["data_cadastro"], format="%Y-%m-%d", errors="coerce")
br = pd.to_datetime(limpo["data_cadastro"], format="%d/%m/%Y", errors="coerce")
limpo["data_cadastro"] = iso.fillna(br)
print(f"3. {limpo['data_cadastro'].isna().sum()} datas não convertidas")

# 4. categorias
limpo["perfil_investidor"] = limpo["perfil_investidor"].str.strip().str.title()
print("4. perfis:", sorted(limpo["perfil_investidor"].dropna().unique()))

# 5. booleano
limpo["ativo"] = limpo["ativo"].astype(str).str.strip().str.lower().map(
    {"sim": True, "nao": False, "1": True, "0": False}
)
print("5. valores de 'ativo':", limpo["ativo"].unique())

# 6. idades impossíveis
invalidas = (~limpo["idade"].between(18, 110)).sum()
limpo.loc[~limpo["idade"].between(18, 110), "idade"] = np.nan
print(f"6. {invalidas} idades impossíveis viraram NaN")

print()
limpo.info()

Dois detalhes que decidem se o resultado está certo:

- **a ordem das substituições** no item 2 (remover o ponto de milhar **antes** de trocar
  a vírgula por ponto);
- **passar `format=` explicitamente** no item 3. Deixar o pandas adivinhar entre
  `01/02/2024` (1º de fevereiro) e `01/02/2024` (2 de janeiro, no padrão americano) é
  pedir um erro que não avisa.

---

## Exercício 14 (código) — Desafio — perguntas de negócio


Com a base de clientes **limpa** do exercício anterior, responda com código:

1. qual perfil concentra o maior patrimônio total?
2. qual a idade mediana de cada perfil?
3. qual o aporte mensal mediano por estado, considerando apenas estados com pelo menos 20
   clientes?
4. clientes ativos têm patrimônio mediano maior que os inativos? De quanto é a diferença?

Escreva, em uma célula de texto ou em comentários, uma conclusão de duas ou três frases
para o gerente da corretora.

In [ ]:
# Reproduzindo a limpeza para esta célula ser independente
limpo = clientes.drop_duplicates().copy()
texto = (
    limpo["patrimonio_investido"].astype(str).str.strip()
    .str.replace("R$", "", regex=False).str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)
limpo["patrimonio_investido"] = pd.to_numeric(texto, errors="coerce")
limpo["perfil_investidor"] = limpo["perfil_investidor"].str.strip().str.title()
limpo["ativo"] = limpo["ativo"].astype(str).str.lower().map(
    {"sim": True, "nao": False, "1": True, "0": False}
)
limpo.loc[~limpo["idade"].between(18, 110), "idade"] = np.nan

# 1
print("1. Patrimônio total por perfil:")
print(limpo.groupby("perfil_investidor")["patrimonio_investido"].sum().round(2)
      .sort_values(ascending=False))

# 2
print("\n2. Idade mediana por perfil:")
print(limpo.groupby("perfil_investidor")["idade"].median())

# 3
por_estado = limpo.groupby("estado").agg(
    clientes=("id_cliente", "count"),
    aporte_mediano=("aporte_mensal", "median"),
)
print("\n3. Estados com 20+ clientes:")
print(por_estado[por_estado["clientes"] >= 20].sort_values("aporte_mediano",
                                                          ascending=False).round(2))

# 4
mediana_por_situacao = limpo.groupby("ativo")["patrimonio_investido"].median()
print("\n4. Patrimônio mediano por situação:")
print(mediana_por_situacao.round(2))
diferenca = mediana_por_situacao.get(True, 0) - mediana_por_situacao.get(False, 0)
print(f"   Diferença (ativos - inativos): R$ {diferenca:,.2f}")

Uma conclusão possível:

> *"O perfil Conservador é o mais numeroso, mas o Arrojado concentra patrimônio mediano
> várias vezes maior — a base é larga na base da pirâmide e pesada no topo. A idade
> mediana é parecida entre os perfis, o que sugere que perfil de risco aqui não é
> explicado por idade. A diferença de patrimônio entre clientes ativos e inativos é
> pequena, o que enfraquece a ideia de que os inativos sejam apenas contas pequenas
> abandonadas."*

Repare no filtro de 20 clientes no item 3: sem ele, um estado com dois clientes apareceria
no ranking com a mesma autoridade de um com cem. **Sempre reporte o tamanho do grupo ao
lado da estatística do grupo.**

E note que a conclusão é redigida com cautela — "sugere", "enfraquece a ideia". Isso não é
falta de convicção: é precisão. Os dados sustentam associações, não afirmações causais.

---

## Fim do gabarito

Se você resolveu a maior parte sem consultar, pode seguir para o próximo módulo. Se
consultou muito, vale refazer os exercícios em que travou — desta vez, sem olhar.